# team_agents

> Bandit and random agents for the patch world (Phase 6).

Spec: `specs/multiagent/phase6/`. `PatchMember` records true harvest; baseline learners and `CircuitPatchMember` for the ring.

In [ ]:
#| default_exp team_agents


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from collections import deque

import numpy as np

from explRL.members import CircuitMember
from explRL.multi import Member, Outcome


class PatchMember(Member):
    timeUnit = 's'
    extraNames = ('target', 'harvest', 'nHere', 'patch')

    def event(self, world, t):
        obs = world.observe(self.id)
        target = self.act(obs)
        reward, dt, loc, info = world.apply(self.id, target, t)
        self.learn(obs, target, reward, info)
        return Outcome(dt, info['kind'], loc, reward,
                       (target, info['harvest'], info['nHere'], info['patch']))


class randomAgent(PatchMember):
    def __init__(self, rng):
        self.rng = rng

    def act(self, obs):
        return int(self.rng.integers(len(obs['travel'])))


class banditAgent(PatchMember):
    def __init__(self, K=None, cUCB=1.0, H=10.0):
        self.K, self.cUCB, self.H = K, float(cUCB), float(H)
        self.rates = None

    def _stats(self, obs):
        n = np.array([len(r) for r in self.rates], float)
        mean = np.array([np.mean(r) if len(r) else 0.0 for r in self.rates])
        return mean, n

    def act(self, obs):
        T = obs['travel']
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in T]
        mean, n = self._stats(obs)
        if (n == 0).any():
            idx = np.flatnonzero(n == 0)
            return int(idx[np.argmin(T[idx])])
        score = (mean + self.cUCB * np.sqrt(np.log1p(n.sum()) / n)) * self.H / (self.H + T)
        return int(np.argmax(score))

    def learn(self, obs, target, reward, info):
        if info['kind'] == 'stay':
            self.rates[obs['location']].append(reward / info['dur'])


class CircuitPatchMember(CircuitMember):
    extraNames = CircuitMember.extraNames + ('harvest',)

    def reward(self, world, t, fix):
        h, net = world.fixate(self.id, self.agent.a, t + self.agent.params.tS, fix, t)
        self._harvest, self._lastNet = h, net
        s = np.sign(net)
        return int(s) if s != 0 else 0

    def event(self, world, t):
        out = super().event(world, t)
        return out._replace(extras=out.extras + (self._harvest,))


## Tests

In [ ]:
import numpy as np


In [ ]:
from pathlib import Path
import json, tempfile
from fastcore.test import test_eq, test_close
from explRL.core import loadCircuitAgent
from explRL.envs import ZoneArena
from explRL.members import CircuitMember
from explRL.multi import runTeam, TeamEnv, spawnRngs
from explRL.patchworld import TeamPatchWorld, ringLayout

# AC-1 sanity bridge
CIRC_N = len(CircuitMember.extraNames)
for symmetric in (True, False):
    for seed in (0, 1, 2):
        env = ZoneArena(symmetric=symmetric)
        world = TeamPatchWorld.zoneConfig(symmetric=symmetric)
        world.reset(seed, 1)
        h0 = runTeam([CircuitMember(loadCircuitAgent(seed=seed))], env, 300.0)
        world.reset(seed, 1)
        h1 = runTeam([CircuitPatchMember(loadCircuitAgent(seed=seed))], world, 300.0)
        test_eq(len(h0.records), len(h1.records))
        for a, b in zip(h0.records, h1.records):
            test_eq((a.t, a.dt, a.location, a.reward), (b.t, b.dt, b.location, b.reward))
            test_eq(a.extras[:CIRC_N], b.extras[:CIRC_N])

# AC-2 (golden fixtures after the reward hook and apply(..., t)) runs in the train, prestructured_train and
# circuit_train notebooks.

# AC-8 agents
rng = np.random.default_rng(0)
ra = randomAgent(rng)
obs = {'travel': np.ones(8)}
counts = np.zeros(8)
for _ in range(10_000):
    counts[ra.act(obs)] += 1
expected = 10_000 / 8
se = np.sqrt(expected * (1 - 1/8))
assert np.all(np.abs(counts - expected) <= 3 * se)

ba = banditAgent(K=3, cUCB=0.2, H=10.0)
obs0 = {'travel': np.arange(8, dtype=float) + 1., 'location': 0}
visited = []
for _ in range(8):
    j = ba.act(obs0)
    visited.append(j)
    ba.learn({'location': j}, j, 1.0, {'kind': 'stay', 'dur': 1.0})
test_eq(visited, list(range(8)))                       # unvisited first, nearest first
obs_tie = {'travel': np.array([0., 1., 1., 1.]), 'location': 0}
test_eq(banditAgent().act(obs_tie), 0)
b = banditAgent()
b.act(obs_tie); b.learn({'location': 0}, 0, 1.0, {'kind': 'stay', 'dur': 1.0})
test_eq(b.act(obs_tie), 1)                             # equal travel times: lower index

ring8 = TeamPatchWorld(ringLayout(8), nPatches=1, nDanger=0, model='fixed', intake=(1.0,),
                       patchSites=[3], dangerSites=[], Tmove=None, tStay=1.0, c=1.0)
ring8.reset(0, 1)
agent = banditAgent(cUCB=0.2, H=10.0)
team = runTeam([agent], ring8, 600.0)
stays = [r for r in team.records if r.kind == 'stay' and r.t >= 450]
share = sum(1 for r in stays if r.location == 3) / max(len(stays), 1)
assert share > 0.8

baK = banditAgent(K=2)
for _ in range(5):
    baK.act({'travel': np.array([1., 1.]), 'location': 0})
    baK.learn({'location': 0}, 0, 1.0, {'kind': 'stay', 'dur': 1.0})
    baK.learn({'location': 0}, 0, 2.0, {'kind': 'stay', 'dur': 1.0})
    baK.learn({'location': 0}, 0, 3.0, {'kind': 'stay', 'dur': 1.0})
test_eq(len(baK.rates[0]), 2)

# AC-10 determinism, JSONL, TeamEnv replay
def patch_world():
    "Ring of 16, sites drawn by the seed, moving patches, noisy harvest: every world generator is used."
    return TeamPatchWorld(ringLayout(16), nPatches=2, nDanger=2, intake=(2.0, 2.0), Tmove=10.0, noise=0.2)

def run_patch_team(seed=0, n=2):
    "Mixed team: random agents (even index) and bandits (odd index)."
    w = patch_world()
    rngs = spawnRngs(seed + 1000, n)
    ms = [randomAgent(r) if i % 2 == 0 else banditAgent() for i, r in enumerate(rngs)]
    w.reset(seed, n)
    hist = runTeam(ms, w, 30.0)
    return w, hist, ms

def same_ledger(a, b):
    test_eq(len(a), len(b))
    for ra, rb in zip(a, b):
        test_eq(len(ra), len(rb))
        for x, y in zip(ra, rb):
            if isinstance(x, float) and np.isnan(x):
                assert np.isnan(y)
            else:
                test_eq(x, y)

w1, h1, _ = run_patch_team(42, 4)
w2, h2, _ = run_patch_team(42, 4)
same_ledger(w1.ledger, w2.ledger)
test_eq(h1.records, h2.records)
w3, _, _ = run_patch_team(42, 6)                       # more agents: same patch sites (separate generators)
test_eq(w3.site.tolist(), w1.site.tolist())

with tempfile.TemporaryDirectory() as d:
    p = h1.toJSONL(Path(d) / 't.jsonl')
    row = json.loads(p.read_text().splitlines()[0])
    test_eq(set(row['extras']), set(PatchMember.extraNames))

w, hist, ms = run_patch_team(7, 4)
actions = [r.extras[0] for r in hist.records]
w2 = patch_world()
env = TeamEnv(w2, 4, 30.0)
env.reset(7)
ai = 0
while (i := env.next_agent()) is not None:
    env.step(int(actions[ai]))
    ai += 1
same_ledger(w2.ledger, w.ledger)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
